<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
موضوع‌بندی
</font>
</h1>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
مقدمه و صورت مسئله
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
به یک تمرین صنعتی از  کاربرد یادگیری ماشین در پردازش زبان طبیعی (NLP) خوش آمدید.  در این تمرین داده‌های واقعی وب فارسی که توسط پلتفرم <a href="https://www.yektanet.com/">یکتانت</a> پالایش و جمع‌آوری شده در اختیار ما قرار گرفته است. هدف تمرین؛ ساخت یک مدل یادگیری ماشین است که با توجه متن‌های موجود در یک پیوند (Link) نظیر <i>عنوان</i>، <i>توضیحات</i>، <i>محتوای متنی [کامل]</i> و غیره بتواند دسته‌ی موضوعی آن سند را پیش‌بینی کند. به‌عنوان مثال اگر پیوندی از یک سایت خبری با عنوان «<i>کیهان کلهر جایزه مرد سال موسیقی جهان را دریافت کرد</i>» داشته باشیم، مدل ما باید پیش‌بینی کند که این مطلب مرتبط با موضوع «موسیقی» است. البته در این مثال ما تنها از ویژگی <i>عنوان</i> یاد کردیم، در حالی‌که می‌توان از متن <i>توضیحات</i> یا <i>محتوای متنی</i> هم کمک گرفت. 

</br>
توجه داشته باشید برای آن‌که بتوانید از الگوریتم‌های معرفی‌شده در کالج جهت کار روی داده‌های متنی استفاده کنید نیاز است حداقل با یکی از روش‌های تعبیه‌سازی (Embedding) آشنا شده باشید تا بتوانید هر متن را به یک بردار ویژگی عددی تبدیل کنید.

</font>
</p>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
وارد کردن کتابخانه‌های مورد نیاز
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    ابتدا کتابخانه‌های مورد نیازتان را وارد کنید.
</font>
</p>

In [1]:
import numpy as np
import pandas as pd

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
معرفی مجموعه‌داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    هر نمونه از این مجموعه‌داده با ویژگی‌هایی که در جدول زیر شرح داده شده همراه است. ستون <code>category</code> متغیر هدف مسئله است که موضوع محتوا را نشان می‌دهد. 
</font>
</p>
<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
|ستون|توضیحات|
|:------:|:---:|
|<code>category</code>| موضوع (متغیر هدف) |
|<code>description</code>| توضیحات |
|<code>text_content</code>| محتوای متنی |
|<code>title</code>| عنوان |
|<code>h1</code>| محتوای تگ <code>h1</code> صفحه |
|<code>h2</code>|محتوای تگ <code>h2</code> صفحه  |
|<code>url</code>| آدرس پیوند|
|<code>domain</code>|دامنه‌ی وب‌سایت |
|<code>id</code>|آیدی پیوند|

</font>
</div>
</center>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
خواندن مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    در ابتدا نیاز است فایل‌های مجموعه‌داده را بخوانید. نمونه‌های آموزشی در فایل <code>yektanet_train.csv</code> و نمونه‌های آزمون که باید موضوع آن‌ها را پیش‌بینی کنید در فایل <code>yektanet_test.csv</code> ذخیره شده‌اند. اگر لازم دانستید می‌توانید به دلخواه خود بخشی از مجموعه‌ی آموزشی را به عنوان مجموعه‌ی اعتبارسنجی نیز جدا کنید.
</font>
</p>

In [54]:
train = pd.read_csv('../data/yektanet_train.csv')

In [55]:
test = pd.read_csv('../data/yektanet_test.csv')

In [56]:
test.isna().sum()

description       3
text_content      0
title             0
h1               33
h2              118
url               0
domain            1
id                0
dtype: int64

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
پیش‌پردازش و مهندسی ویژگی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
در هنگام کار با داده‌های متنی، پیش‌پردازش داده‌ها به کمک تکنیک‌های پردازش زبان طبیعی یکی از مراحل بسیار تاثیرگذار در یادگیری مدل و عملکرد نهایی است.
در تمرین «کامنت‌کاوی» فصل دسته‌بندی با چندین تکنیک رایج پیش‌پردازش خصوصاً برای زبان فارسی آشنا شدید. در این قسمت نیز می‌توانید تابعی بنویسید که یک رشته (<code>string</code>) را گرفته، اصلاحات موردنظر شما رو روی متن اعمال کرده و در نهایت نتیجه را با فرمت یک رشته (<code>string</code>) خروجی دهد. بررسی و تحلیل کلمات موجود در مجموعه‌داده از نظر تعداد رخداد نیز می‌تواند شما را در پیش‌پردازش بهتر یاری کند.
</font>
</p>


In [57]:
X = train['title'] + train['text_content']
X_test = test['title'] + test['text_content']
Y = train['category']

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
در هنگام تعبیه‌سازی به کمک کتابخانه‌ی <code>scikit-learn</code>
از آنجا که ممکن است توکن‌ساز (Tokenizer) مورد استفاده در این کتابخانه مناسب زبان فارسی نباشد، بهتر است از توکن‌سازهای مناسب این زبان استفاده کنیم. کافیست تابعی بنویسید که یک رشته (<code>string</code>) را گرفته و به کمک کتابخانه‌ی موردنظر شما (مثل <code>word_tokenize</code> در کتابخانه‌ی هضم) توکن‌های آن را جدا کند. خروجی تابع لیستی از توکن‌ها خواهد بود.
</font>
</p>


In [58]:
from hazm import Normalizer, word_tokenize, stopwords_list
import string ,re 

normalizer = Normalizer()
stop_words = set(stopwords_list())

punctuations = set(string.punctuation + "؟،؛«»…")

def tokenizer(text):
    text = normalizer.normalize(text)
    text = text.replace("\n", " ")
    tokens = word_tokenize(text)

    persian_to_english_digits = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")
    text = text.translate(persian_to_english_digits)
    
    filtered = ' '.join([word for word in tokens if (word not in stop_words and word not in punctuations and not word.isdigit() and not re.match(r'^[A-Za-z]+$', word))])    
    return filtered


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
توجه داشته باشید انتخاب این‌که از کدام ویژگی‌ها (متن‌ها) به‌عنوان ورودی الگوریتم استفاده کنید بر عهده‌ی خودتان است. می‌توانید تنها از یک ستون مثل <code>title</code> استفاده کنید یا می‌توانید به‌نحوی متن یا حتی بردار ویژگی هر ستون را با همدیگر ترکیب کنید. همچنین فراموش نکنید که ستون متغیر هدف یعنی <code>category</code> نیاز به کدگذاری دارد.
</font>
</p>


In [59]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
Y = encoder.fit_transform(Y)

In [60]:
X = X.apply(tokenizer)
X_test = X_test.apply(tokenizer)

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
مدل‌سازی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
اکنون وقت آن رسیده که الگوریتم یادگیری ماشین موردنظر خود را بر روی داده‌های آموزشی اجرا کنید. در انتخاب الگوریتم کاملاً آزاد هستید. برای این بخش می‌توانید ابتدا هر ورودی متن را به کمک تکنیک‌های معرفی‌شده در درسنامه‌های این فصل (مثل Bag-of-Word یا Tf-idf) به بردارهای ویژگی عددی تبدیل کنید. سپس این بردارها را همراه با لیست برچسب‌های صحیح به الگوریتم بدهید تا مدل آموخته شود. علاوه بر این می‌توانید تمام این مراحل را در یک خط لوله‌ (<code>Pipeline</code>) نیز تعریف کنید.
</font>
</p>

In [61]:
import pickle

with open('../stopwords.json', 'rb') as f:
    stopwords = pickle.load(f)
stopwords = stopwords.tolist()


In [62]:
from sklearn.model_selection import train_test_split

X_train ,X_val ,y_train ,y_val = train_test_split(X,Y,test_size=.2,random_state=42)

In [81]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

tfidf_pipeline = Pipeline([('tfidf', TfidfVectorizer(sublinear_tf=True)),
                ('clf-svm', LinearSVC(loss='hinge', penalty='l2', max_iter=5000))])

In [82]:
tfidf_pipeline.fit(X_train,y_train)
y_pred = tfidf_pipeline.predict(X_val)

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
معیار ارزیابی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    معیاری که برای ارزیابی عملکرد مدل انتخاب کرده‌ایم، <code>F1-score</code> نام دارد و از رویکرد میانگین‌گیری وزن‌دار (<code dir=ltr>average='weighted'</code>) استفاده می‌شود.
    <br>
    پیشنهاد می‌شود با توجه به این معیار، عملکرد مدل خود را بر روی مجموعه‌ی آموزش یا اعتبارسنجی ارزیابی کنید و طبق نتایج به‌دست‌آمده پارامترهای مدل خود را بهتر تنظیم کنید.
</font>
</p>

In [83]:
from sklearn.metrics import f1_score

print('f1_score :',f1_score(y_pred,y_val,average='weighted'))

f1_score : 0.8235731204353104


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
 پیش‌بینی برای داده تست و خروجی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    پس از مهندسی ویژگی و مدل‌سازی، الگوریتمی دارید که می‌تواند شما را از متغیرهای مستقل به متغیر هدف برساند.
    <br>
    از این مدل برای پیش‌بینی نمونه‌های موجود در مجموعه‌ی آزمون استفاده کنید و نتایج را در یک دیتافریم تک‌ستونه با نام <code>submission</code> و در قالب زیر آماده کنید. توجه داشته باشید که ترتیب پیش‌بینی شما اهمیت دارد یعنی به عنوان مثال پیش‌بینی مدل برای نمونه‌ی آزمون <code>m</code> ام را باید در ردیف <code>m</code> ام این دیتافریم ذخیره کنید.
</font>
</p>

<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
|ستون|توضیحات|
|:------:|:---:|
|<code>category</code>|پیش‌بینی مدل شما (از جنس رشته)|
    
</font>
</div>
</center>

In [73]:
result = encoder.inverse_transform(tfidf_pipeline.predict(X_test))
submission = pd.DataFrame({'category':result})
submission.shape

(417, 1)

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
<b>سلول جواب‌ساز</b>
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    برای ساخته‌شدن فایل <code>result.zip</code> سلول زیر را اجرا کنید. توجه داشته باشید که پیش از اجرای سلول زیر تغییرات اعمال شده در نت‌بوک را ذخیره کرده باشید (<code>ctrl+s</code>) تا در صورت نیاز به پشتیبانی امکان بررسی کد شما وجود داشته باشد.
</font>
</p>

In [74]:
import zipfile
import joblib
import os

if not os.path.exists(os.path.join(os.getcwd(), 'text_categorization.ipynb')):
    %notebook -e text_categorization.ipynb


def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

submission.to_csv('submission.csv', index=False)
file_names = ['text_categorization.ipynb', 'submission.csv']
compress(file_names)

File Paths:
['text_categorization.ipynb', 'submission.csv']


<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
💭 اضافه: ابرِ کلمات (Word Cloud)
</font>
</h2>

<center>
<img src="wordcloud.png">
</center>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
یکی از کتابخانه‌های بسیار جالب مرتبط با متن در پایتون، <a href="https://github.com/amueller/word_cloud"><code>WordCloud</code></a> نام دارد. این کتابخانه به شما کمک می‌کند تا ابری از پرتکرارترین توکن‌های موجود در یک مجموعه‌متن را به شکلی زیبا به تصویر بکشید. خوشبختانه نسخه‌ی فارسی این کتابخانه نیز وجود دارد که می‌توانید از <a href="https://github.com/alihoseiny/word_cloud_fa">این لینک</a> به صفحه‌ی گیت‌هاب آن مراجعه کنید. حتی می‌توانید به‌صورت دلخواه تصویری را تعیین کنید تا نمایش نهایی توکن‌ها تداعی‌کننده‌ی تصویر موردنظر شما باشد.
</font>
</p>

In [ ]:
cloud_text = ''
for text in X_train:
    cloud_text += text + ' '

In [ ]:
from wordcloud_fa import WordCloudFa

wc = WordCloudFa(width=1200, height=800, persian_normalize=True, include_numbers=False, max_words=120, background_color='white', min_font_size=10, max_font_size=100)
word_cloud = wc.generate(cloud_text)
image = word_cloud.to_image()
image.show()
image.save('wordcloud.png')

Defaulting to user installation because normal site-packages is not writeable
  Using cached wordcloud_fa-0.1.10-py3-none-any.whl.metadata (11 kB)
  Using cached arabic_reshaper-3.0.0-py3-none-any.whl.metadata (12 kB)
  Using cached python_bidi-0.4.2-py2.py3-none-any.whl.metadata (4.6 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [1 lines of output]
      ERROR: Can not execute `setup.py` since setuptools is not available in the build environment.
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


ModuleNotFoundError: No module named 'wordcloud_fa'

<h4 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
<b>راهنمایی</b>
</font>
</h4>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    ۱. از تکنیک n-gram کمک بگیرید.
    <br>
    ۲. توازن مجموعه‌داده را بررسی کنید.
    <br>
    ۳. در پیش‌پردازش خود می‌توانید حذف حروف اضافه و اعداد، حذف کلمات توقف، نرمال‌سازی و... را آزمایش کنید.
</font>
</p>